# Sprint 3: SOTA Upgrade (Memory Efficient Pipeline)

Notebook ini mengimplementasikan perbaikan besar-besaran (Sprint 3) untuk menyamakan performa dengan project SOTA MSCNN-BiLSTM-AE.

## Key Improvements:
1. **Preprocessing Baru:** MinMax Scaler (0-1) yang bersih, fit hanya pada Benign Train.
2. **Clipping:** Menangani outlier ekstrim di data Test dengan clipping ke [0, 1].
3. **Memory Efficient:** Menggunakan `tf.data.Dataset` generator untuk streaming data (anti-OOM).
4. **Modular:** Menggunakan script terpisah untuk preprocessing yang konsisten.

In [ ]:
# @title Colab Bootstrap (with Drive Persistence)
# Jalankan cell ini jika di Google Colab untuk setup environment
from pathlib import Path
import os
import subprocess
import sys
import shutil

COLAB_BOOTSTRAP_ENABLE = True
COLAB_REPO_URL = "https://github.com/akwancakra/nids-cnn-lstm-autoencoder.git"
COLAB_BRANCH = "feat/sprint3-sota-upgrade"
COLAB_REPO_DIR = Path("/content/nids-cnn-lstm-autoencoder")

# Drive Persistence Config
COLAB_DRIVE_MOUNT = Path("/content/drive")
COLAB_PROJECT_DRIVE_ROOT = COLAB_DRIVE_MOUNT / "MyDrive/nids-cnn-lstm-autoencoder"

def _is_colab_runtime() -> bool:
    try:
        import google.colab  # noqa: F401
        return True
    except Exception:
        return False

def _run_shell(cmd: list[str], cwd: Path | None = None) -> None:
    print(f"[CMD] {' '.join(cmd)}")
    subprocess.run(cmd, cwd=str(cwd) if cwd else None, check=True)

def _ensure_symlink_dir(repo_path: Path, drive_target: Path) -> None:
    """Symlink repo folder to drive folder for persistence."""
    drive_target.mkdir(parents=True, exist_ok=True)
    
    if repo_path.is_symlink():
        if repo_path.resolve() == drive_target.resolve():
            print(f"[INFO] Symlink OK: {repo_path} -> {drive_target}")
            return
        repo_path.unlink()
    elif repo_path.exists():
        if any(repo_path.iterdir()):
            print(f"[WARN] Folder exists and not empty: {repo_path}. Skipping link.")
            return
        repo_path.rmdir()
        
    repo_path.parent.mkdir(parents=True, exist_ok=True)
    os.symlink(str(drive_target), str(repo_path), target_is_directory=True)
    print(f"[INFO] Symlink created: {repo_path} -> {drive_target}")

if _is_colab_runtime() and COLAB_BOOTSTRAP_ENABLE:
    from google.colab import drive
    drive.mount(str(COLAB_DRIVE_MOUNT))
    
    if not COLAB_REPO_DIR.exists():
        _run_shell(["git", "clone", "-b", COLAB_BRANCH, COLAB_REPO_URL, str(COLAB_REPO_DIR)])
    else:
        _run_shell(["git", "fetch", "--all"], cwd=COLAB_REPO_DIR)
        _run_shell(["git", "checkout", COLAB_BRANCH], cwd=COLAB_REPO_DIR)
        _run_shell(["git", "pull", "origin", COLAB_BRANCH], cwd=COLAB_REPO_DIR)
        
    # PERSISTENCE: Link Sprint 3 Data & Models to Drive
    # 1. Processed Data
    _ensure_symlink_dir(
        COLAB_REPO_DIR / "data/research/sprint3_upgrade/processed",
        COLAB_PROJECT_DRIVE_ROOT / "data/research/sprint3_upgrade/processed"
    )
    # 2. Models & Results
    _ensure_symlink_dir(
        COLAB_REPO_DIR / "research/sprint3_upgrade/models",
        COLAB_PROJECT_DRIVE_ROOT / "research/sprint3_upgrade/models"
    )
    _ensure_symlink_dir(
        COLAB_REPO_DIR / "research/sprint3_upgrade/results",
        COLAB_PROJECT_DRIVE_ROOT / "research/sprint3_upgrade/results"
    )
        
    # Set Working Directory
    os.chdir(COLAB_REPO_DIR)
    print(f"[INFO] Working Directory set to: {os.getcwd()}")
else:
    print("Not in Colab or Bootstrap disabled.")

In [ ]:
# Setup Project Root & Imports
import os
import sys
from pathlib import Path
import yaml
import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt
import glob
from tqdm.notebook import tqdm

# Deteksi Project Root (Fallback mechanism)
try:
    # Jika di Colab dan sudah chdir, getcwd() adalah root
    PROJECT_ROOT = Path(os.getcwd()).resolve()
    
    # Validasi sederhana: cek folder research ada atau tidak
    if not (PROJECT_ROOT / "research").exists():
        # Coba naik satu level (jika notebook dijalankan lokal dari folder notebooks/)
        if (PROJECT_ROOT.parent / "research").exists():
            PROJECT_ROOT = PROJECT_ROOT.parent
        else:
            # Fallback hardcoded untuk Colab jika chdir gagal
            PROJECT_ROOT = Path("/content/nids-cnn-lstm-autoencoder")
except Exception:
    PROJECT_ROOT = Path(".")

print(f"Project Root: {PROJECT_ROOT}")
sys.path.append(str(PROJECT_ROOT))

# Load Config Sprint 3
config_path = PROJECT_ROOT / "research/sprint3_upgrade/config/experiment_v1.yaml"

if config_path.exists():
    with open(config_path, 'r') as f:
        config = yaml.safe_load(f)
    print(f"Experiment: {config['experiment_name']}")
    print(f"Config Loaded: {config_path}")
else:
    print(f"ERROR: Config not found at {config_path}")
    print("Make sure you have cloned the repo and switched to the correct branch.")

### 1.1 EDA Data Raw (Opsional)
Inspeksi singkat file CSV mentah untuk melihat kolom, missing values, dan distribusi label.

In [ ]:
import pandas as pd
from pathlib import Path
import glob
import os
from IPython.display import display

def pick_first_existing(paths):
    for p in paths:
        if p is None:
            continue
        p = Path(p)
        if p.exists():
            return p
    return None

raw_train_dir = None
raw_test_dir = None

if 'raw_train' in locals() and 'raw_test' in locals():
    raw_train_dir = Path(raw_train)
    raw_test_dir = Path(raw_test)

if raw_train_dir is None and 'config' in locals() and 'paths' in config:
    if 'raw_train' in config['paths']:
        raw_train_dir = PROJECT_ROOT / config['paths']['raw_train']
    if 'raw_test' in config['paths']:
        raw_test_dir = PROJECT_ROOT / config['paths']['raw_test']

if raw_train_dir is None:
    raw_train_dir = pick_first_existing([
        "/content/drive/MyDrive/nids-data/raw/CIC-IDS2017",
        PROJECT_ROOT / "data" / "raw" / "CIC-IDS2017"
    ])
if raw_test_dir is None:
    raw_test_dir = pick_first_existing([
        "/content/drive/MyDrive/nids-data/raw/CSE-CIC-IDS2018",
        PROJECT_ROOT / "data" / "raw" / "CSE-CIC-IDS2018"
    ])

def inspect_raw_dir(raw_dir, name):
    print(f"\n--- RAW {name} ---")
    if raw_dir is None or not Path(raw_dir).exists():
        print("Raw folder not found.")
        return
    raw_dir = Path(raw_dir)
    files = sorted(glob.glob(str(raw_dir / "*.csv")))
    if not files:
        files = sorted(glob.glob(str(raw_dir / "**" / "*.csv"), recursive=True))
    if not files:
        print(f"No CSV files found at {raw_dir}")
        return
    print(f"Folder: {raw_dir}")
    print(f"Files: {len(files)}")
    sample_file = files[0]
    print(f"Sample file: {os.path.basename(sample_file)}")
    try:
        df = pd.read_csv(sample_file, nrows=20000)
    except Exception:
        df = pd.read_csv(sample_file, nrows=5000, engine="python")
    print(f"Shape: {df.shape}")
    display(df.head(5))
    print("Missing (top 10):")
    print(df.isna().sum().sort_values(ascending=False).head(10))
    label_col = "Label" if "Label" in df.columns else ("label" if "label" in df.columns else None)
    if label_col:
        print(f"Label distribution ({label_col}):")
        print(df[label_col].value_counts().head(20))

inspect_raw_dir(raw_train_dir, "CIC-IDS2017 (Train)")
inspect_raw_dir(raw_test_dir, "CSE-CIC-IDS2018 (Test)")


### 1.2 Ringkasan Label CIC vs CSE (Raw)
Menampilkan total benign/attack dan distribusi per serangan dari dataset mentah.

In [ ]:
import pandas as pd
from pathlib import Path
import glob
from IPython.display import display

def find_label_column(sample_file):
    cols = pd.read_csv(sample_file, nrows=0, low_memory=False).columns.tolist()
    clean_map = {str(c).strip().lower(): c for c in cols}
    return clean_map.get("label")

def count_labels_in_dir(raw_dir, dataset_name, chunksize=200000):
    print(f"\n--- LABEL SUMMARY {dataset_name} ---")
    if raw_dir is None or not Path(raw_dir).exists():
        print("Raw folder not found.")
        return None, None
    raw_dir = Path(raw_dir)
    files = sorted(glob.glob(str(raw_dir / "*.csv")))
    if not files:
        files = sorted(glob.glob(str(raw_dir / "**" / "*.csv"), recursive=True))
    if not files:
        print(f"No CSV files found at {raw_dir}")
        return None, None
    label_col = find_label_column(files[0])
    if not label_col:
        print("Label column not found.")
        return None, None
    counts = {}
    for f in files:
        for chunk in pd.read_csv(f, usecols=[label_col], chunksize=chunksize, low_memory=False):
            vc = chunk[label_col].fillna("UNKNOWN").astype(str).value_counts()
            for k, v in vc.items():
                counts[k] = counts.get(k, 0) + int(v)
    if not counts:
        print("No label counts found.")
        return None, None
    total = int(sum(counts.values()))
    benign_total = int(sum(v for k, v in counts.items() if str(k).strip().upper() == "BENIGN"))
    attack_total = int(total - benign_total)
    overall_df = pd.DataFrame([
        {"dataset": dataset_name, "benign": benign_total, "attack": attack_total, "total": total}
    ])
    attack_rows = [
        {"dataset": dataset_name, "label": k, "count": int(v)}
        for k, v in counts.items()
        if str(k).strip().upper() != "BENIGN"
    ]
    attack_df = (
        pd.DataFrame(attack_rows).sort_values(by="count", ascending=False)
        if attack_rows
        else pd.DataFrame(columns=["dataset", "label", "count"])
    )
    display(overall_df)
    display(attack_df)
    return overall_df, attack_df

count_labels_in_dir(raw_train_dir, "CIC-IDS2017 (Train)")
count_labels_in_dir(raw_test_dir, "CSE-CIC-IDS2018 (Test)")


## 1. Data Inspection (EDA)
Menganalisis distribusi data mentah (RAW) atau processed yang tersedia.
Cell ini akan mencoba mendeteksi data secara otomatis di path standar.

In [ ]:
# @title Data Inspection & Class Distribution
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter
from tqdm.notebook import tqdm
import numpy as np
import os
import glob
from pathlib import Path

# Default values to prevent NameError
train_files = []
test_files = []

# --- AUTO DETECT DATA PATHS ---
# Coba deteksi path data processed (hasil run sebelumnya)
try:
    base_processed = None
    if 'config' in locals() and 'paths' in config:
        if 'processed_data' in config['paths']:
            base_processed = PROJECT_ROOT / config['paths']['processed_data']
        elif 'train_data' in config['paths']:
            base_processed = Path(config['paths']['train_data']).parent

    if base_processed is None:
        candidates = [
            Path("data/processed"),
            Path("../data/processed"),
            Path("/content/nids-cnn-lstm-autoencoder/data/research/sprint3_upgrade/processed"),
            Path("/content/nids-mscnn-bilstm-autoencoder/data/processed")
        ]
        for p in candidates:
            if p.exists():
                base_processed = p
                break

    if base_processed:
        print(f"[INFO] Inspecting data at: {base_processed}")
        train_files = sorted(glob.glob(str(base_processed / "train" / "*.npz")))
        test_files = sorted(glob.glob(str(base_processed / "test" / "*.npz")))
        
        if not train_files:
             train_files = sorted(glob.glob(str(base_processed / "**" / "train" / "*.npz"), recursive=True))
        if not test_files:
             test_files = sorted(glob.glob(str(base_processed / "**" / "test" / "*.npz"), recursive=True))
    else:
        print("[WARN] Processed data directory not found yet. Run preprocessing first or check paths.")
        
except Exception as e:
    print(f"[ERROR] Path detection failed: {e}")

def analyze_distribution(files, set_name="Dataset"):
    print(f"\n--- Analyzing {set_name} ---")
    if not files:
        print("No files found.")
        return

    try:
        with np.load(files[0], allow_pickle=True) as data:
            print(f"Sample File: {os.path.basename(files[0])}")
            print(f"Keys: {list(data.keys())}")
            for k in data.keys():
                obj = data[k]
                print(f"  {k}: shape={obj.shape}, dtype={obj.dtype}")
    except Exception as e:
        print(f"Error reading sample file: {e}")
        return

    label_counts = Counter()
    total_samples = 0
    
    scan_limit = min(len(files), 100) 
    print(f"Scanning {scan_limit} files (sample) for label distribution...")
    
    for f in tqdm(files[:scan_limit], desc=f"Scanning {set_name}"):
        try:
            with np.load(f, allow_pickle=True) as data:
                y = data['y'] if 'y' in data else (data['Y'] if 'Y' in data else None)
                if y is not None:
                    if len(y.shape) > 0:
                        label_counts.update(y)
                    else:
                        label_counts[y.item()] += 1
                    total_samples += len(y) if len(y.shape) > 0 else 1
        except: pass
        
    scale_factor = len(files) / scan_limit
    estimated_total = int(total_samples * scale_factor)
    
    print(f"Scanned Samples: {total_samples}")
    print(f"Estimated Total Samples: {estimated_total}")
    print(f"Label Distribution (Scanned): {dict(label_counts)}")
    
    if label_counts:
        plt.figure(figsize=(10, 5))
        keys = list(label_counts.keys())
        vals = list(label_counts.values())
        sns.barplot(x=keys, y=vals)
        plt.title(f"Label Distribution (Sampled) - {set_name}")
        plt.xlabel("Label (0=Benign, 1=Attack)")
        plt.ylabel("Count")
        plt.grid(axis='y', linestyle='--', alpha=0.7)
        plt.show()

if train_files:
    analyze_distribution(train_files, "Train Set (Benign Expected)")
else:
    print("Train files not found. (Maybe preprocessing hasn't run yet?)")

if test_files:
    analyze_distribution(test_files, "Test Set (Mixed Expected)")
else:
    print("Test files not found.")


## 2. Preprocessing (SOTA Style)
Jalankan script preprocessing baru yang menerapkan MinMax(0-1) dan Clipping.

In [ ]:
# Run Preprocessing Script (If data not exists)
processed_dir = PROJECT_ROOT / config['paths']['processed_data']
train_dir = processed_dir / "train"
test_dir = processed_dir / "test"

# FORCE RE-RUN PREPROCESSING TO FIX LABEL BUG
RUN_PREPROCESSING = False 

# --- AUTO DETECT & REUSE LOGIC ---
# Check if processed data already exists (locally or via symlink)
if train_dir.exists() and any(train_dir.iterdir()):
    print("[INFO] Processed data found! But RUN_PREPROCESSING is forced to True.")
    # RUN_PREPROCESSING = False # Disabled to force re-run

# Adjust Raw Paths for Colab Support
if os.path.exists("/content/drive/MyDrive/nids-data/raw"):
    print("[INFO] Colab Drive Raw Data Detected.")
    raw_train = Path("/content/drive/MyDrive/nids-data/raw/CIC-IDS2017")
    raw_test = Path("/content/drive/MyDrive/nids-data/raw/CSE-CIC-IDS2018")
else:
    # Local fallback or repo path
    raw_train = PROJECT_ROOT / config['paths']['raw_train']
    raw_test = PROJECT_ROOT / config['paths']['raw_test']
# -------------------------------------------------

if RUN_PREPROCESSING:
    print("Running Preprocessing Script... (This may take a while)")
    script_path = PROJECT_ROOT / "research/sprint3_upgrade/scripts/preprocess_sota.py"
    
    # Check raw data
    if not raw_train.exists():
        print(f"Error: Raw train path {raw_train} not found!")
        print("Please check config or mount drive correctly.")
    else:
        cmd = f'python "{script_path}" --raw_train "{raw_train}" --raw_test "{raw_test}" --output_dir "{processed_dir}" --seq_len {config["preprocessing"]["sequence_length"]} --stride {config["preprocessing"]["stride"]}'
        print(f"Executing: {cmd}")
        !{cmd}
else:
    print("Skipping Preprocessing (Data already exists).")
    print(f"Data Location: {processed_dir}")

In [ ]:
# @title Verify New Test Set Distribution
import numpy as np
import glob
import os

test_files = sorted(glob.glob(str(test_dir / "*.npz")))
if test_files:
    print(f"Found {len(test_files)} test files. Checking the first one...")
    sample_file = test_files[0]
    with np.load(sample_file, allow_pickle=True) as data:
        y = data['y'] if 'y' in data else (data['Y'] if 'Y' in data else None)
        if y is not None:
            unique, counts = np.unique(y, return_counts=True)
            dist = dict(zip(unique, counts))
            print(f"Label Distribution in {os.path.basename(sample_file)}: {dist}")
            if 0 in dist and 1 in dist:
                print("✅ SUCCESS: Both Benign (0) and Attack (1) are present!")
            else:
                print("❌ ERROR: Missing classes. Check preprocessing logic.")
else:
    print("No test files found. Preprocessing might have failed.")

## 3. Data Pipeline (Memory Efficient)
Menggunakan `tf.data.Dataset` generator agar hemat RAM.

In [ ]:
# @title 1. Data Pipeline (Flexible: Memory Efficient vs Speed Optimized)
# Pilih mode sesuai kapasitas RAM Colab Anda.

# --- KONFIGURASI MODE ---
PIPELINE_MODE = "SPEED_OPTIMIZED"  # Options: "MEMORY_EFFICIENT" (12GB RAM) or "SPEED_OPTIMIZED" (25GB+ RAM)
# -----------------------

import tensorflow as tf
import glob
import numpy as np
import os

print(f"[INFO] Pipeline Mode: {PIPELINE_MODE}")

def npz_generator(files):
    """
    Generator yang membaca file .npz dari list file satu per satu dan yield batch data.
    """
    if not files:
        # print(f"Warning: No files provided") # Silent warning to avoid spam in logs
        return
        
    for f in files:
        try:
            with np.load(f, allow_pickle=True) as data:
                # Handle keys
                X = data['X'] if 'X' in data else (data['x'] if 'x' in data else None)
                
                if X is None:
                    continue
                    
                # Yield per sample (tf.data will batch it later)
                for i in range(len(X)):
                    # Autoencoder: Input = Target (X, X)
                    yield X[i], X[i] # Target is X for AE
        except Exception as e:
            print(f"Error reading {f}: {e}")

def create_dataset(files, batch_size=256, shuffle=True, input_shape=(10, 77)):
    """
    Membuat tf.data.Dataset dari list file shards.
    """
    # Tentukan output signature
    output_signature = (
        tf.TensorSpec(shape=input_shape, dtype=tf.float32), # Input X
        tf.TensorSpec(shape=input_shape, dtype=tf.float32)  # Target X (Autoencoder)
    )
    
    dataset = tf.data.Dataset.from_generator(
        lambda: npz_generator(files),
        output_signature=output_signature
    )
    
    # --- LOGIKA MODE ---
    if PIPELINE_MODE == "SPEED_OPTIMIZED":
        # Cache ke RAM setelah pembacaan pertama.
        # Epoch 1: Lambat (Read Disk + Cache). Epoch 2+: Cepat (Read RAM).
        # HATI-HATI: Butuh RAM besar (~20-30GB untuk full dataset)
        dataset = dataset.cache()
        shuffle_buffer = 50000 if shuffle else 1000
    else:
        # Memory Efficient (Streaming)
        # Tidak ada cache, baca disk terus menerus. Hemat RAM tapi lambat.
        shuffle_buffer = 10000 if shuffle else 1000
    
    if shuffle:
        dataset = dataset.shuffle(buffer_size=shuffle_buffer)
        
    dataset = dataset.batch(batch_size)
    dataset = dataset.prefetch(tf.data.AUTOTUNE)
    return dataset

# Helper untuk menghitung steps_per_epoch
def count_samples(files):
    total = 0
    if not files:
        print("Warning: No files to count.")
        return 0
        
    print(f"Counting samples from {len(files)} files...")
    for f in files:
        try:
            with np.load(f, allow_pickle=True) as data:
                key = 'X' if 'X' in data else 'x'
                if key in data:
                    total += data[key].shape[0]
        except: pass
    return total

print("Data Pipeline Ready.")

## 4. Build & Train Model


In [ ]:
from tensorflow.keras.layers import Input, Conv1D, MaxPooling1D, Concatenate, Bidirectional, LSTM, Dropout, Flatten, Dense, RepeatVector, UpSampling1D
from tensorflow.keras.models import Model

# Setup Paths & Config
model_save_dir = PROJECT_ROOT / config['paths']['model_save_dir']
model_save_dir.mkdir(parents=True, exist_ok=True)
best_model_path = model_save_dir / "best_model.keras"
final_model_path = model_save_dir / "final_model.keras"

FORCE_RETRAIN = False # Set True to force retrain even if model exists

def build_model(input_shape, encoding_dim=16):
    inputs = Input(shape=input_shape)
    # Encoder
    # (10, 77)
    conv1 = Conv1D(32, 3, activation='relu', padding='same')(inputs)
    pool1 = MaxPooling1D(2, padding='same')(conv1) # (5, 32)
    conv2 = Conv1D(16, 3, activation='relu', padding='same')(pool1)
    # Skip pool2 agar tidak ganjil/sulit di upsample balik ke 10
    lstm1 = LSTM(64, return_sequences=True)(conv2) # (5, 64)
    dropout1 = Dropout(0.2)(lstm1)
    flatten = Flatten()(dropout1)
    encoded = Dense(encoding_dim, activation='relu')(flatten)
    
    # Decoder
    repeat = RepeatVector(5)(encoded) # (5, 16)
    lstm2 = LSTM(64, return_sequences=True)(repeat) # (5, 64)
    dropout2 = Dropout(0.2)(lstm2)
    conv3 = Conv1D(16, 3, activation='relu', padding='same')(dropout2) # (5, 16)
    upsample1 = UpSampling1D(2)(conv3) # (10, 16)
    decoded = Conv1D(input_shape[1], 3, activation='sigmoid', padding='same')(upsample1) # (10, 77)
    
    autoencoder = Model(inputs, decoded)
    autoencoder.compile(optimizer='adam', loss='mse')
    return autoencoder

# --- LOGIC: LOAD OR TRAIN ---

if not FORCE_RETRAIN and (best_model_path.exists() or final_model_path.exists()):
    print("✅ Pre-trained model found! Loading model...")
    load_path = best_model_path if best_model_path.exists() else final_model_path
    
    try:
        model = tf.keras.models.load_model(str(load_path))
        print(f"Model loaded from: {load_path}")
        model.summary()
        
        # We STILL need Validation Data for threshold calculation later
        print("\n[INFO] Loading Validation Data for Threshold Calculation...")
        BATCH_SIZE = config['training']['batch_size']
        if 'train_dir' not in locals():
            train_dir = PROJECT_ROOT / config['paths']['processed_data'] / "train"
            
        all_files = sorted(glob.glob(str(train_dir / "*.npz")))
        # Use simple split for now just to get val data
        split_idx = int(len(all_files) * (1 - config['training']['validation_split']))
        val_files = all_files[split_idx:]
        
        if val_files:
            val_ds = create_dataset(val_files, batch_size=BATCH_SIZE, shuffle=False)
            print(f"Validation Dataset Loaded ({len(val_files)} files). Ready for Thresholding.")
        else:
             print("Warning: No validation files found.")

    except Exception as e:
        print(f"❌ Error loading model: {e}")
        print("Falling back to training...")
        FORCE_RETRAIN = True

if FORCE_RETRAIN or ('model' not in locals()):
    print("⚡ Starting Training Process...")
    
    BATCH_SIZE = config['training']['batch_size']

    # 1. Dataset Setup
    if 'train_dir' not in locals():
        train_dir = PROJECT_ROOT / config['paths']['processed_data'] / "train"
        
    all_files = sorted(glob.glob(str(train_dir / "*.npz")))
    split_idx = int(len(all_files) * (1 - config['training']['validation_split']))
    train_files = all_files[:split_idx]
    val_files = all_files[split_idx:]

    train_samples = count_samples(train_files)
    val_samples = count_samples(val_files)

    print(f"Total Train Samples: {train_samples}")
    print(f"Total Val Samples: {val_samples}")
    
    if train_samples > 0:
        # Create Datasets
        train_ds = create_dataset(train_files, batch_size=BATCH_SIZE, shuffle=True).repeat()
        val_ds = create_dataset(val_files, batch_size=BATCH_SIZE, shuffle=False)
        
        train_steps = max(1, train_samples // BATCH_SIZE)
        val_steps = max(1, val_samples // BATCH_SIZE)
        
        # Build Model
        input_shape = (10, 77)
        model = build_model(input_shape, encoding_dim=config['model']['encoding_dim'])
        model.summary()
        
        # Callbacks
        callbacks = [
            tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=config['training']['patience'], restore_best_weights=True, verbose=1),
            tf.keras.callbacks.ModelCheckpoint(str(best_model_path), save_best_only=True, monitor='val_loss')
        ]
        
        # Fit
        history = model.fit(
            train_ds,
            epochs=config['training']['epochs'],
            steps_per_epoch=train_steps,
            validation_data=val_ds,
            validation_steps=val_steps,
            callbacks=callbacks,
            verbose=1
        )
        
        model.save(str(final_model_path))
        print(f"Model saved to {final_model_path}")
        
        # Plot History
        plt.plot(history.history['loss'], label='Train')
        plt.plot(history.history['val_loss'], label='Val')
        plt.xlabel('Epochs')
        plt.ylabel('Loss (MSE)')
        plt.legend()
        plt.show()
    else:
        print("❌ No training data found. Cannot train.")

In [ ]:
# @title 4.1 Determine Threshold & Visualization
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
from tqdm.notebook import tqdm

def find_threshold(model, val_ds, percentile=95):
    print("Calculating reconstruction errors on Validation Set (Benign)...")
    val_errors = []
    
    # Calculate MSE for all validation samples
    for batch_x, _ in tqdm(val_ds, desc="Calculating Threshold"):
        recon = model.predict(batch_x, verbose=0)
        # MSE per sample
        mse = np.mean(np.square(batch_x - recon), axis=(1, 2))
        val_errors.extend(mse)
    
    val_errors = np.array(val_errors)
    
    # Determine Threshold
    threshold_val = np.percentile(val_errors, percentile)
    print(f"\nConfiguration:")
    print(f"  percentile: {percentile}%")
    print(f"  threshold : {threshold_val:.6f}")
    
    # Plot Distribution
    plt.figure(figsize=(10, 5))
    sns.histplot(val_errors, bins=50, kde=True, color='blue', label='Benign (Val) Errors')
    plt.axvline(threshold_val, color='red', linestyle='--', label=f'Threshold ({percentile}%)')
    plt.title(f"Reconstruction Error Distribution (Benign) - Threshold: {threshold_val:.5f}")
    plt.xlabel("Mean Squared Error (MSE)")
    plt.ylabel("Frequency")
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.show()
    
    return threshold_val

# Execute
if 'val_ds' in locals() and 'model' in locals():
    # Gunakan percentile dari config jika ada, default 95
    pct = config['thresholding']['percentile'] if 'config' in locals() else 95
    threshold = find_threshold(model, val_ds, percentile=pct)
    print(f"✅ Threshold set to: {threshold}")
else:
    print("❌ Validation dataset (val_ds) or Model not found. Run training first.")

## Evaluation (Dual-Set)
Evaluasi model pada dua dataset:
1. **CSE-CIC-IDS2018 (Wajib):** Test set utama (Zero-Day Attack).
2. **CIC-IDS2017 (Opsional):** Test set in-domain (jika tersedia).

Set variable `RUN_ALL_EVAL = True` untuk menjalankan keduanya.

In [ ]:
# @title Dual Evaluation (CSE + CIC) - Fast Mode
RUN_ALL_EVAL = True

# --- CONFIG: EVALUATION SPEED ---
EVAL_SAMPLE_FRACTION = 0.2  # 0.2 = Run on 20% of data (Faster). 1.0 = Full Data.
# --------------------------------

import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import os
import glob
import tensorflow as tf
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
from tqdm.notebook import tqdm

def evaluate_performance(y_true, y_pred, name="Test Set", color='Blues'):
    y_true = y_true.astype(int)
    y_pred = y_pred.astype(int)
    
    acc = accuracy_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred, zero_division=0)
    rec = recall_score(y_true, y_pred, zero_division=0)
    f1 = f1_score(y_true, y_pred, zero_division=0)
    
    print(f"\n[{name}] Performance:")
    print(f"  Accuracy : {acc:.4f}")
    print(f"  Precision: {prec:.4f}")
    print(f"  Recall   : {rec:.4f}")
    print(f"  F1-Score : {f1:.4f}")
    
    cm = confusion_matrix(y_true, y_pred)
    plt.figure(figsize=(5, 4))
    sns.heatmap(cm, annot=True, fmt='d', cmap=color)
    plt.title(f'Confusion Matrix ({name})')
    plt.ylabel('True Label')
    plt.xlabel('Predicted Label')
    plt.show()
    return f1

def npz_generator_with_label(files):
    for f in files:
        try:
            with np.load(f, allow_pickle=True) as data:
                X = data['X'] if 'X' in data else (data['x'] if 'x' in data else None)
                y = data['y'] if 'y' in data else (data['Y'] if 'Y' in data else None)
                
                if X is not None and y is not None:
                    for i in range(len(X)):
                        yield X[i], y[i]
        except Exception: pass

def create_eval_dataset(files, batch_size=256):
    # Specialized dataset for evaluation returning (X, y) instead of (X, X)
    output_signature = (
        tf.TensorSpec(shape=(10, 77), dtype=tf.float32),
        tf.TensorSpec(shape=(), dtype=tf.int32) # Label is scalar int
    )
    
    dataset = tf.data.Dataset.from_generator(
        lambda: npz_generator_with_label(files),
        output_signature=output_signature
    )
    return dataset.batch(batch_size).prefetch(tf.data.AUTOTUNE)

def calculate_errors_and_predict(dataset, model, threshold, steps=None):
    all_errors = []
    all_y_true = []
    
    desc = "Predicting"
    prog_bar = tqdm(dataset, total=steps, desc=desc) if steps else tqdm(dataset, desc=desc)
    
    # Counter untuk limit sample
    batch_count = 0
    
    for batch_x, batch_y in prog_bar:
        # Predict Reconstruction
        recon = model.predict(batch_x, verbose=0)
        # Calculate MSE per sample
        mse = np.mean(np.square(batch_x - recon), axis=(1, 2))
        
        all_errors.extend(mse)
        all_y_true.extend(batch_y.numpy())
        
        # Stop jika sudah mencapai steps (untuk fast mode)
        batch_count += 1
        if steps and batch_count >= steps:
            break
    
    all_errors = np.array(all_errors)
    all_y_true = np.array(all_y_true)
    
    # Thresholding
    y_pred = (all_errors > threshold).astype(int)
    
    return all_y_true, y_pred, all_errors

# --- 1. Evaluate on CSE-CIC-IDS2018 (Main Test Set) ---
print("="*40)
print("1. EVALUATION: CSE-CIC-IDS2018 (Cross-Domain / Zero-Day)")
print("="*40)

if 'threshold' not in locals():
    print("⚠️ Threshold not defined. Using default 0.05")
    threshold = 0.05

# FORCE RELOAD CSE DATASET in (X, y) mode
cse_ds = None
cse_steps = None
test_files = []

# Locate files
if 'test_dir' in locals() and test_dir.exists():
     test_files = sorted(glob.glob(str(test_dir / "*.npz")))
elif 'config' in locals():
     base_test = PROJECT_ROOT / config['paths']['test_data']
     if base_test.exists():
         test_files = sorted(glob.glob(str(base_test / "*.npz")))

# Fallback Search
if not test_files:
    candidates = [
        PROJECT_ROOT / "data/processed/test",
        Path("/content/nids-cnn-lstm-autoencoder/data/research/sprint3_upgrade/processed/test"),
        Path("/content/nids-mscnn-bilstm-autoencoder/data/processed/test")
    ]
    for c in candidates:
        if c.exists():
            test_files = sorted(glob.glob(str(c / "*.npz")))
            if test_files: break

if test_files:
    # --- FAST MODE LOGIC ---
    # Ambil sample files secara acak jika jumlah file banyak
    # Tapi demi konsistensi, mending ambil slice depan.
    # Namun data kita dishuffle saat create npz?
    # Jika ya, ambil slice aman. Jika sequence, hati-hati.
    
    # OPSI A: Gunakan semua file, tapi batasi STEPS di dataset loop
    # Ini menjamin kita menscan file secara berurutan sampai limit tercapai.
    
    total_samples = count_samples(test_files)
    BATCH_SIZE = config['training']['batch_size'] if 'config' in locals() else 256
    
    full_steps = max(1, total_samples // BATCH_SIZE)
    # Apply Sample Fraction
    cse_steps = int(full_steps * EVAL_SAMPLE_FRACTION)
    cse_steps = max(1, cse_steps) # Minimal 1 step
    
    print(f"Found {len(test_files)} files. Total Samples: {total_samples}")
    print(f"Running FAST EVALUATION on {EVAL_SAMPLE_FRACTION*100}% Data.")
    print(f"Processing {cse_steps} batches (approx {cse_steps*BATCH_SIZE} samples)...")
    
    # Create DATASET with LABELS specifically for evaluation
    cse_ds = create_eval_dataset(test_files, batch_size=BATCH_SIZE)
    
    y_true_cse, y_pred_cse, errors_cse = calculate_errors_and_predict(cse_ds, model, threshold, cse_steps)
    evaluate_performance(y_true_cse, y_pred_cse, name="CSE-CIC-IDS2018", color='Blues')
else:
    print("❌ CSE-CIC-IDS2018 Dataset files not found.")


# --- 2. Evaluate on CIC-IDS2017 (In-Domain Test Set) ---
if RUN_ALL_EVAL:
    print("\n" + "="*40)
    print("2. EVALUATION: CIC-IDS2017 (In-Domain Reference)")
    print("="*40)
    
    cic_test_path = None
    possible_roots = [
        PROJECT_ROOT / "data" / "processed",
        Path("/content/nids-cnn-lstm-autoencoder/data/research/sprint3_upgrade/processed"),
        Path("/content/nids-mscnn-bilstm-autoencoder/data/processed")
    ]
    
    for root in possible_roots:
        candidate = root / "test_cic"
        if candidate.exists():
            cic_test_path = candidate
            print(f"Found CIC Test Path at: {candidate}")
            break
            
    if cic_test_path:
        cic_files = sorted(glob.glob(str(cic_test_path / "*.npz")))
        if cic_files:
            total_samples_cic = count_samples(cic_files)
            full_steps_cic = max(1, total_samples_cic // BATCH_SIZE)
            
            # Apply Sample Fraction
            cic_steps = int(full_steps_cic * EVAL_SAMPLE_FRACTION)
            cic_steps = max(1, cic_steps)

            print(f"Running FAST EVALUATION on {EVAL_SAMPLE_FRACTION*100}% CIC Data.")
            print(f"Processing {cic_steps} batches (approx {cic_steps*BATCH_SIZE} samples)...")
            
            # Create Dataset
            cic_ds = create_eval_dataset(cic_files, batch_size=BATCH_SIZE)
            
            y_true_cic, y_pred_cic, errors_cic = calculate_errors_and_predict(cic_ds, model, threshold, cic_steps)
            evaluate_performance(y_true_cic, y_pred_cic, name="CIC-IDS2017", color='Greens')
        else:
            print("❌ CIC Test folder empty.")
    else:
        print("ℹ️ CIC-IDS2017 Test Data folder 'test_cic' not found.")

## 5. Comprehensive Experiment Summary & Review
Ringkasan lengkap hasil eksperimen untuk evaluasi mandiri dan reviewer.
Mencakup metrik kunci, interpretasi performa, dan catatan anomali.

In [ ]:
# @title Generate Comprehensive Report
import pandas as pd
from IPython.display import display, Markdown
import datetime

def generate_report():
    print("\n" + "="*50)
    print(f"EXPERIMENT REPORT | {datetime.datetime.now().strftime('%Y-%m-%d %H:%M')}")
    print("="*50 + "\n")
    
    report_data = []
    
    # 1. Model & Training Config
    if 'config' in locals():
        train_cfg = config.get('training', {})
        model_cfg = config.get('model', {})
        print("1. CONFIGURATION")
        print(f"   - Model Type: {model_cfg.get('type', 'Custom Autoencoder')}")
        print(f"   - Epochs: {train_cfg.get('epochs', '?')}")
        print(f"   - Batch Size: {train_cfg.get('batch_size', '?')}")
        print(f"   - Threshold Percentile: P{config.get('thresholding', {}).get('percentile', '?')}")
        print("-"*30)
    
    # 2. Performance Metrics (Collect from recent run)
    # We try to grab variables from global scope if they exist
    # Note: This relies on the variables being present in memory from previous cells
    
    metrics_found = False
    md_table = "| Dataset | Accuracy | Precision | Recall | F1-Score | Status |\n| :--- | :--- | :--- | :--- | :--- | :--- |\n"
    
    # Check CSE Results (Usually y_true, y_pred are from last run)
    # Ideally we should store them in a dict, but let's try to capture from 'y_true' if it was CSE
    # A more robust way is if we saved results to a dict/file. 
    # Let's assume the user just ran the Evaluation cell.
    
    # Placeholder for logic: In a real report, we'd read from the saved 'evaluation_results.yaml' 
    # or check the last computed metric variables.
    
    print("2. PERFORMANCE SUMMARY")
    print("   (See Markdown table below for details)")
    
    # 3. Interpretation & Reviewer Notes
    interpretation = """
### 3. Interpretation & Self-Evaluation

#### A. Zero-Day Detection Capability (CSE-CIC-IDS2018)
- **Recall Analysis:** High Recall (>85%) indicates the model effectively identifies unknown attacks. Low Recall implies overfitting to 'normal' patterns of 2017.
- **Precision Analysis:** High Precision indicates low False Alarms. If Precision is low, the threshold might be too tight (flagging benign anomalies as attacks).

#### B. Domain Generalization (CIC-IDS2017)
- If available, high scores here confirm the model learned the baseline 'normal' correctly without underfitting.

#### C. Potential Issues to Check
- **Overfitting:** If Train Loss << Val Loss.
- **Threshold Sensitivity:** Did a small change in percentile drastically change F1-Score?
- **Data Leakage:** Ensure 'Benign' samples in Test set were NOT in Training set.
"""
    display(Markdown(interpretation))
    
    print("\n" + "="*50)
    print("END OF REPORT")
    print("="*50)

generate_report()